# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook:** c16 - Timber Stock Dataset Generation  
**Author:** Jasper Cluistra  
**Last Updated:** 2026-05-17

---

### Goal
Generate the timber stock datasets used by the optimizer when assigning cross-sections to structural members. New stock is drawn from a parameterised catalogue covering standard softwood sections; reclaimed stock is sampled from a synthetic donor building.

### Inputs
- Stock generation parameters (`c16_generation_timber`)
- Representative beam length statistics (`representative_beam_statistics.json` from c12_15)

### Outputs
| File | Description |
|------|-------------|
| `new_timber_v2.csv` | New softwood stock only |
| `complete_timber_v2.csv` | New + reclaimed stock combined — used by the optimizer |
| `complete_timber_small.csv` | Small mixed subset for fast development runs |

# 1. Stock generation
Generates new and reclaimed stock for both donor buildings (A and B) in a single run, so all output files share **the same new stock**. Saves four files:
- `complete_timber_new.csv` — new stock only (shared baseline)
- `complete_timber_A.csv` — new + reclaimed donor A
- `complete_timber_B.csv` — new + reclaimed donor B

In [ ]:
import pandas as pd
import config
from c16_generation_timber import generate_new_stock, generate_reclaimed_stock

efficient = True  # True = minimum transport distance; False = realistic randomised distances

# ── Generate New Stock (once — shared across all output files) ────────────────
df_new_stock = generate_new_stock(efficient)
display(df_new_stock.head(3))

# ── Generate Reclaimed Stock — Donor A and B ──────────────────────────────────
# "A" — 3-storey Dutch residential (hardcoded spans: 1800–4500 mm)
# "B" — commercial/industrial (spans derived from structure's min/max length statistics)
df_reclaimed_A = generate_reclaimed_stock(donor_building="A")
df_reclaimed_B = generate_reclaimed_stock(donor_building="B")

for label, df_rs in [("A", df_reclaimed_A), ("B", df_reclaimed_B)]:
    print(f"\nDonor {label}: {len(df_rs)} elements  |  "
          f"length {df_rs['Length'].min():.0f}–{df_rs['Length'].max():.0f} mm  |  "
          f"mean {df_rs['Length'].mean():.0f} mm")

# ── Combine: same new stock base + each donor's reclaimed pool ────────────────
df_combined_A = pd.concat([df_new_stock, df_reclaimed_A], ignore_index=True)
df_combined_B = pd.concat([df_new_stock, df_reclaimed_B], ignore_index=True)

# ── Save ──────────────────────────────────────────────────────────────────────
files = {
    "complete_timber_new_v2.csv": df_new_stock,
    "complete_timber_A_v2.csv":   df_combined_A,
    "complete_timber_B_v2.csv":   df_combined_B,
}
for filename, df in files.items():
    df.to_csv(config.TIMBER_STOCK_PATH / filename, index=False, sep=';')
    print(f"Saved '{filename}'  ({len(df)} elements  |  "
          f"NS={( df['State'] == 0).sum()}  RS={(df['State'] == 1).sum()})")

print(f"\nNew stock is identical across A, B, and new — {len(df_new_stock)} NS elements shared.")

Config System loaded successfully, Code running locally from thesis_generative_timber and Data is connected to OneDrive 2.2 - 2.4.


Generating catalog...
  primary=306 beam types, tail=115 beam types

Length source stats (new stock):
  primary      (p5–p95):  (1800, 2100, 2400, 2700, 3000, 3300, 3600, 3900, 4200)
  short tail   (min→p5):  (600, 900, 1200, 1500)
  long tail    (p95→max): (4500, 4800, 5100, 5400, 5700, 6000)

Generated lengths — tail_margin_mm=300
  Sections per tail length (graduated 6..17):
    short tail: {600: 6, 900: 10, 1200: 13, 1500: 17}
    long tail:  {4500: 17, 4800: 15, 5100: 13, 5400: 10, 5700: 8, 6000: 6}

New stock generated: 421 elements total
  primary=306, short_tail=46, long_tail=69

Elements per length:
    600 mm: 6
    900 mm: 10
    1200 mm: 13
    1500 mm: 17
    1800 mm: 34
    2100 mm: 34
    2400 mm: 34
    2700 mm: 34
    3000 mm: 34
    3300 mm: 34
    3600 mm: 34
    3900 mm: 34
    4200 mm: 34
    4500 mm: 17
    4800 mm: 15
    5100 mm: 1

,Member_ID,State,Length,Depth,Width,Length_Category,Availability_Probability,f_mk,f_tk,E_modulus_eff,E_modulus_005,f_vk,f_c0k,k_density,mean_density,Origin_Country,Transport_Dist,EmissionFactor
0,NS_00000,0,1800.0,100.0,38.0,primary,1.0,24.0,14.0,11000.0,7400.0,2.5,21.0,350.0,420.0,Germany,293.46,0.1799
1,NS_00001,0,1800.0,100.0,50.0,primary,1.0,24.0,14.0,11000.0,7400.0,2.5,21.0,350.0,420.0,Netherlands,57.35,0.1702
2,NS_00002,0,1800.0,100.0,63.0,primary,1.0,24.0,14.0,11000.0,7400.0,2.5,21.0,350.0,420.0,Netherlands,51.86,0.1719


Donor building raw pool: 132 elements across 3 floors
  edge_beam: 24 raw -> 19 survived
  primary_beam: 36 raw -> 27 survived
  secondary_joist: 45 raw -> 35 survived
  short_joist: 27 raw -> 22 survived
Total surviving elements: 103

Reclaimed stock summary:
  Total elements: 103
  Length mean:    3074.7 mm
  Length std:     958.0 mm
  Length min:     1784 mm
  Length max:     4500 mm
  Unique lengths: 54

  By role:
    edge_beam: 19 elements, 2687–2700 mm, section 90x190 mm
    primary_beam: 27 elements, 4487–4500 mm, section 110x230 mm
    secondary_joist: 35 elements, 2986–3000 mm, section 70x190 mm
    short_joist: 22 elements, 1784–1799 mm, section 60x170 mm
Donor building raw pool: 198 elements across 3 floors
  floor_joist: 45 raw -> 36 survived
  long_rafter: 63 raw -> 51 survived
  mid_beam: 54 raw -> 41 survived
  short_purlin: 36 raw -> 27 survived
Total surviving elements: 155

Reclaimed stock summary:
  Total elements: 155
  Length mean:    3744.2 mm
  Length std:     1

## 2. Section combinations
Inspect all unique Depth × Width combinations present in the new stock catalogue. Uncomment to run.

In [2]:
# depth_width_combinations = (
#     df_new_stock[['Depth', 'Width']]
#     .drop_duplicates()
#     .sort_values(by=['Depth', 'Width'])
#     .reset_index(drop=True)
# )
#
# print(f"\n{'#':<6} {'D (mm)':<12} {'W (mm)':<12}")
# print("-" * 30)
# for idx, (_, row) in enumerate(depth_width_combinations.iterrows(), 1):
#     print(f"{idx:<6} {int(row['Depth']):<12} {int(row['Width']):<12}")
# print("-" * 30)
# print(f"Total combinations: {len(depth_width_combinations)}")

## 3. Development subset
Generates a small mixed stock file (`complete_timber_small.csv`) for fast development and testing runs in the optimizer. Adjust `total_elements` and `reclaimed_ratio` as needed.

In [3]:
import config
from c16_generation_timber import generate_mixed_stock_subset

# Small development subsets — generated independently (not required to share NS pool)
for donor in ["A", "B"]:
    df_small = generate_mixed_stock_subset(
        total_elements    = 20,
        reclaimed_ratio   = 0.3,
        random_state      = 38,
        allow_replacement = True,
    )
    filename = f"complete_timber_small_{donor}.csv"
    df_small.to_csv(config.TIMBER_STOCK_PATH / filename, index=False, sep=';')
    print(f"Saved '{filename}'  ({len(df_small)} elements)")


Generating catalog...
  primary=306 beam types, tail=115 beam types

Length source stats (new stock):
  primary      (p5–p95):  (1800, 2100, 2400, 2700, 3000, 3300, 3600, 3900, 4200)
  short tail   (min→p5):  (600, 900, 1200, 1500)
  long tail    (p95→max): (4500, 4800, 5100, 5400, 5700, 6000)

Generated lengths — tail_margin_mm=300
  Sections per tail length (graduated 6..17):
    short tail: {600: 6, 900: 10, 1200: 13, 1500: 17}
    long tail:  {4500: 17, 4800: 15, 5100: 13, 5400: 10, 5700: 8, 6000: 6}

New stock generated: 421 elements total
  primary=306, short_tail=46, long_tail=69

Elements per length:
    600 mm: 6
    900 mm: 10
    1200 mm: 13
    1500 mm: 17
    1800 mm: 34
    2100 mm: 34
    2400 mm: 34
    2700 mm: 34
    3000 mm: 34
    3300 mm: 34
    3600 mm: 34
    3900 mm: 34
    4200 mm: 34
    4500 mm: 17
    4800 mm: 15
    5100 mm: 13
    5400 mm: 10
    5700 mm: 8
    6000 mm: 6

Donor building raw pool: 132 elements across 3 floors
  edge_beam: 24 raw -> 20 sur